# Outline
In this notebook, we will perform model training with hyper parameter tunning

In [1]:
import numpy as np
import pandas as pd
import fastparquet

In [5]:
X_train = pd.read_csv('../data/interim/X_train_transformed.csv',index_col=0)
X_test = pd.read_csv('../data/interim/X_test_transformed.csv',index_col=0)
y_train = pd.read_csv('../data/interim/y_train.csv')
y_test = pd.read_csv('../data/interim/y_test.csv')

In [6]:
X_train.head()

,HomePlanet_Europa,HomePlanet_Mars,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,...,FoodCourt,ShoppingMall,Spa,VRDeck,TotalBill,CryoSleep,PassengerId,Cabin,Name,Num
4696,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.801184,1.060544,0.000000,0.000000,0.092013,False,5007_01,F/951/S,Gramus Watie,951.0
5946,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.030236,1.403965,0.766971,1.423170,0.028545,False,6308_01,G/1017/P,Jord Cofferson,1017.0
227,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.000000,0.000000,0.000000,0.000000,-0.903270,True,0244_01,NaN,Froos Sad,NaN
3950,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,-0.903270,True,4216_01,B/134/P,Rotan Dratembid,134.0
7674,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.000000,0.782453,1.479169,0.367282,0.006878,False,8191_01,G/1320/S,Gracy Gaington,1320.0


In [3]:
X_train.head()

,HomePlanet_Europa,HomePlanet_Mars,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,...,Side,CryoSleep,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Num,TotalBill
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.423756,1.672589,0.802434,1.060544,0.000000,0.000000,0.424782,0.092074
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.059170,0.000000,1.031843,1.403965,0.766971,1.417181,0.463522,0.028436
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.762119,0.000000,0.000000,0.000000,0.000000,0.000000,0.146339,-0.905876
3,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.445798,0.000000,0.000000,0.000000,0.000000,0.000000,-0.704127,-0.905876
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,-0.653835,0.000000,0.000000,0.782453,1.479169,0.365736,0.614107,0.006710


In [4]:
import optuna
from sklearn.model_selection import KFold,cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [5]:
y_train

,Transported
0,False
1,False
2,True
3,True
4,False
...,...
5819,True
5820,False
5821,False
5822,False


In [50]:
def Multiple_Objective_Function(trial):
    predictor = trial.suggest_categorical('predictor', ['rf', 'lr', 'svc', 'xgb', 'light'])

    if predictor == 'rf':
        rf_n_estimators = trial.suggest_int('rf_n_estimators', 50, 200)
        rf_max_depth = trial.suggest_int('rf_max_depth', 3, 32)
        rf_min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 10)
        
        model = RandomForestClassifier(
            n_estimators=rf_n_estimators, 
            max_depth=rf_max_depth, 
            min_samples_split=rf_min_samples_split,
            random_state=42
        )

    elif predictor == 'lr':
        lr_c = trial.suggest_float('lr_c', 1e-4, 1e2, log=True)
        lr_solver = trial.suggest_categorical('lr_solver', ['lbfgs', 'liblinear'])
        
        model = LogisticRegression(
            C=lr_c, 
            solver=lr_solver, 
            max_iter=2000, # Increased to help 'lbfgs' converge
            random_state=42
        )

    elif predictor == 'svc':
        svc_c = trial.suggest_float('svc_c', 1e-3, 1e2, log=True)
        svc_kernel = trial.suggest_categorical('svc_kernel', ['linear', 'rbf'])
        svc_gamma = trial.suggest_categorical('svc_gamma', ['scale', 'auto'])
        
        model = SVC(
            C=svc_c, 
            kernel=svc_kernel, 
            gamma=svc_gamma,
            random_state=42
        )

    elif predictor == 'xgb':
        xgb_n_estimators = trial.suggest_int('xgb_n_estimators', 50, 200)
        xgb_lr = trial.suggest_float('xgb_lr', 1e-3, 0.3, log=True)
        xgb_max_depth = trial.suggest_int('xgb_max_depth', 3, 9)
        xgb_subsample = trial.suggest_float('xgb_subsample', 0.5, 1.0)
        
        model = XGBClassifier(
            n_estimators=xgb_n_estimators, 
            learning_rate=xgb_lr, 
            max_depth=xgb_max_depth,
            subsample=xgb_subsample,
            random_state=42
        )

    elif predictor == 'light':
        light_n_estimators = trial.suggest_int('light_n_estimators', 50, 200)
        light_lr = trial.suggest_float('light_lr', 1e-3, 0.3, log=True)
        light_num_leaves = trial.suggest_int('light_num_leaves', 20, 100)
        light_subsample = trial.suggest_float('light_subsample', 0.5, 1.0)
        
        model = LGBMClassifier(
            n_estimators=light_n_estimators, 
            learning_rate=light_lr, 
            num_leaves=light_num_leaves,
            subsample=light_subsample,
            random_state=42,
            verbose=-1 # Silences LightGBM terminal warnings
        )

    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = cross_validate(
        model,
        X_train,
        y_train['Transported'],
        scoring='accuracy',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1
    )

    train_score_mean = cv_results['train_score'].mean()
    val_score_mean = cv_results['test_score'].mean()

    trial.set_user_attr('train_score_mean', train_score_mean)
    trial.set_user_attr('train_score_std', cv_results['train_score'].std())
    trial.set_user_attr('test_score_std', cv_results['test_score'].std())
    trial.set_user_attr('overfitting_gap', train_score_mean - val_score_mean)
    

    return (val_score_mean)

In [45]:
multiple_study = optuna.create_study(direction='maximize',study_name='Model_Selection')

[I 2026-07-27 10:53:06,810] A new study created in memory with name: Model_Selection


In [46]:
multiple_study.optimize(Multiple_Objective_Function,n_trials=30,n_jobs=-1)

[I 2026-07-27 10:53:13,891] Trial 2 finished with value: 0.8044290075660369 and parameters: {'predictor': 'rf', 'rf_n_estimators': 147, 'rf_max_depth': 24, 'rf_min_samples_split': 6}. Best is trial 2 with value: 0.8044290075660369.
[I 2026-07-27 10:53:15,656] Trial 3 finished with value: 0.7925808592540153 and parameters: {'predictor': 'xgb', 'xgb_n_estimators': 192, 'xgb_lr': 0.21475612809870417, 'xgb_max_depth': 7, 'xgb_subsample': 0.5446043235538645}. Best is trial 2 with value: 0.8044290075660369.
[I 2026-07-27 10:53:22,066] Trial 1 finished with value: 0.7893192041650073 and parameters: {'predictor': 'xgb', 'xgb_n_estimators': 118, 'xgb_lr': 0.0015197707181302134, 'xgb_max_depth': 6, 'xgb_subsample': 0.5630478033756084}. Best is trial 2 with value: 0.8044290075660369.
[I 2026-07-27 10:53:24,790] Trial 4 finished with value: 0.7955013789950296 and parameters: {'predictor': 'light', 'light_n_estimators': 80, 'light_lr': 0.01818234814963735, 'light_num_leaves': 87, 'light_subsample':

In [47]:
df = multiple_study.trials_dataframe()

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype          
---  ------                       --------------  -----          
 0   number                       30 non-null     int64          
 1   value                        30 non-null     float64        
 2   datetime_start               30 non-null     datetime64[ns] 
 3   datetime_complete            30 non-null     datetime64[ns] 
 4   duration                     30 non-null     timedelta64[ns]
 5   params_light_lr              16 non-null     float64        
 6   params_light_n_estimators    16 non-null     float64        
 7   params_light_num_leaves      16 non-null     float64        
 8   params_light_subsample       16 non-null     float64        
 9   params_lr_c                  4 non-null      float64        
 10  params_lr_solver             4 non-null      object         
 11  params_predictor             30 no

In [52]:
df.groupby("params_predictor")['value'].mean().sort_values()

params_predictor
lr       0.765883
svc      0.789749
xgb      0.796909
rf       0.799565
light    0.800760
Name: value, dtype: float64

In [53]:
df.groupby("params_predictor")['user_attrs_overfitting_gap'].mean().sort_values()

params_predictor
lr       0.005709
svc      0.015089
rf       0.081731
xgb      0.097931
light    0.111980
Name: user_attrs_overfitting_gap, dtype: float64

In [55]:
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[False,True]).head()

,number,value,datetime_start,datetime_complete,duration,params_light_lr,params_light_n_estimators,params_light_num_leaves,params_light_subsample,params_lr_c,...,params_xgb_lr,params_xgb_max_depth,params_xgb_n_estimators,params_xgb_subsample,user_attrs_Predictor,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
11,11,0.809581,2026-07-27 10:53:29.921491,2026-07-27 10:53:45.081856,0 days 00:00:15.160365,0.051611,200.0,37.0,0.956350,NaN,...,NaN,NaN,NaN,NaN,light,0.129121,0.009872,0.938702,0.003299,COMPLETE
5,5,0.808722,2026-07-27 10:53:15.665419,2026-07-27 10:53:26.204214,0 days 00:00:10.538795,0.157905,87.0,34.0,0.686391,NaN,...,NaN,NaN,NaN,NaN,light,0.141784,0.007732,0.950507,0.001249,COMPLETE
29,29,0.808207,2026-07-27 10:53:57.657760,2026-07-27 10:54:02.364157,0 days 00:00:04.706397,0.068804,128.0,42.0,0.521537,NaN,...,NaN,NaN,NaN,NaN,light,0.129207,0.008210,0.937414,0.002709,COMPLETE
19,19,0.807006,2026-07-27 10:53:49.157237,2026-07-27 10:53:54.420146,0 days 00:00:05.262909,0.034190,88.0,48.0,0.735647,NaN,...,NaN,NaN,NaN,NaN,light,0.070141,0.007017,0.877146,0.002467,COMPLETE
7,7,0.806492,2026-07-27 10:53:24.795731,2026-07-27 10:53:29.917312,0 days 00:00:05.121581,NaN,NaN,NaN,NaN,NaN,...,0.042071,7.0,110.0,0.787896,xgb,0.088168,0.008320,0.894660,0.001556,COMPLETE


In [7]:

def rf_objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12), 
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 15),
        'max_features': trial.suggest_float('max_features', 0.4, 0.8),
        'max_samples': trial.suggest_float('max_samples', 0.5, 0.9),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    
    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = cross_validate(
        model,
        X_train,
        y_train['Transported'],
        scoring='accuracy',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1
    )

    train_score_mean = cv_results['train_score'].mean()
    val_score_mean = cv_results['test_score'].mean()

    # Log metrics to the Optuna trial
    trial.set_user_attr('train_score_mean', train_score_mean)
    trial.set_user_attr('train_score_std', cv_results['train_score'].std())
    trial.set_user_attr('test_score_std', cv_results['test_score'].std())
    trial.set_user_attr('overfitting_gap', train_score_mean - val_score_mean)
    
    return val_score_mean

def lgb_objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50), 
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 5),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
        'random_state': 42,
        'verbose': -1
    }
    
    model = LGBMClassifier(**param)
    
    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = cross_validate(
        model,
        X_train,
        y_train['Transported'],
        scoring='accuracy',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1
    )

    train_score_mean = cv_results['train_score'].mean()
    val_score_mean = cv_results['test_score'].mean()

    trial.set_user_attr('train_score_mean', train_score_mean)
    trial.set_user_attr('train_score_std', cv_results['train_score'].std())
    trial.set_user_attr('test_score_std', cv_results['test_score'].std())
    trial.set_user_attr('overfitting_gap', train_score_mean - val_score_mean)
    
    return val_score_mean

In [57]:
rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective,n_trials=30)

[I 2026-07-27 11:23:34,787] A new study created in memory with name: no-name-981762dd-461f-424e-82f2-c026c92e3744
[I 2026-07-27 11:23:48,139] Trial 0 finished with value: 0.7867430644661739 and parameters: {'n_estimators': 219, 'max_depth': 4, 'min_samples_split': 6, 'min_samples_leaf': 15, 'max_features': 0.5293756815522709, 'max_samples': 0.6356091084400788}. Best is trial 0 with value: 0.7867430644661739.
[I 2026-07-27 11:23:58,015] Trial 1 finished with value: 0.8011673524770291 and parameters: {'n_estimators': 206, 'max_depth': 11, 'min_samples_split': 29, 'min_samples_leaf': 3, 'max_features': 0.7822456539518257, 'max_samples': 0.560685948371605}. Best is trial 1 with value: 0.8011673524770291.
[I 2026-07-27 11:24:08,081] Trial 2 finished with value: 0.7639061693435394 and parameters: {'n_estimators': 356, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 3, 'max_features': 0.7213186848494784, 'max_samples': 0.6040754324329101}. Best is trial 1 with value: 0.8011673524

In [8]:
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective,n_trials=30)

[I 2026-07-28 09:48:49,471] A new study created in memory with name: no-name-58a92582-3f7f-4ab5-b6d1-e1e5d5edac35
[I 2026-07-28 09:49:08,890] Trial 0 finished with value: 0.804773387608218 and parameters: {'n_estimators': 422, 'learning_rate': 0.0468457412741649, 'max_depth': 3, 'num_leaves': 30, 'min_child_samples': 93, 'reg_alpha': 0.005661152061764648, 'reg_lambda': 0.16466636390464387, 'subsample': 0.8309149782256302, 'subsample_freq': 3, 'colsample_bytree': 0.4106612901567255}. Best is trial 0 with value: 0.804773387608218.
[I 2026-07-28 09:49:14,243] Trial 1 finished with value: 0.8107828562158016 and parameters: {'n_estimators': 380, 'learning_rate': 0.03656924693016446, 'max_depth': 7, 'num_leaves': 49, 'min_child_samples': 39, 'reg_alpha': 0.0219168317426632, 'reg_lambda': 1.824836683717424, 'subsample': 0.877554342097719, 'subsample_freq': 5, 'colsample_bytree': 0.4242808297771443}. Best is trial 1 with value: 0.8107828562158016.
[I 2026-07-28 09:49:15,917] Trial 2 finished w

In [9]:
df = lgb_study.trials_dataframe()

In [62]:
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[False,True]).head()

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,params_subsample_freq,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
12,12,0.813359,2026-07-27 11:33:25.499669,2026-07-27 11:33:28.519826,0 days 00:00:03.020157,0.689940,0.053656,7,22,199,23,1.180334,0.218461,0.508366,2,0.062628,0.005063,0.875987,0.003047,COMPLETE
28,28,0.813015,2026-07-27 11:34:15.476968,2026-07-27 11:34:16.996942,0 days 00:00:01.519974,0.689189,0.056726,6,51,126,12,0.788558,0.276586,0.578150,1,0.024940,0.005082,0.837955,0.001707,COMPLETE
21,21,0.812329,2026-07-27 11:33:50.637661,2026-07-27 11:33:54.173774,0 days 00:00:03.536113,0.417112,0.045305,7,47,233,37,0.098542,0.000829,0.529593,3,0.055202,0.008124,0.867531,0.001838,COMPLETE
29,29,0.811642,2026-07-27 11:34:16.999517,2026-07-27 11:34:18.996091,0 days 00:00:01.996574,0.682716,0.080396,5,66,193,10,0.000118,0.399510,0.601440,1,0.042195,0.003502,0.853838,0.001679,COMPLETE
14,14,0.811642,2026-07-27 11:33:31.465868,2026-07-27 11:33:33.192197,0 days 00:00:01.726329,0.798542,0.050635,7,36,101,18,0.274194,0.001102,0.566529,2,0.034598,0.005906,0.846240,0.002764,COMPLETE


In [10]:
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[False,True]).head()

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,params_subsample_freq,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
12,12,0.812500,2026-07-28 09:49:48.237399,2026-07-28 09:49:52.087151,0 days 00:00:03.849752,0.654714,0.059160,6,30,319,10,3.620272,9.996471,0.719611,4,0.032753,0.002361,0.845252,0.001289,COMPLETE
5,5,0.812329,2026-07-28 09:49:23.020145,2026-07-28 09:49:26.384196,0 days 00:00:03.364051,0.688721,0.025518,6,48,320,13,0.061303,0.739532,0.641192,4,0.033610,0.004267,0.845939,0.001407,COMPLETE
3,3,0.812157,2026-07-28 09:49:15.918736,2026-07-28 09:49:18.585775,0 days 00:00:02.667039,0.572972,0.069102,6,96,250,14,0.001263,0.001217,0.799841,1,0.057864,0.005304,0.870021,0.002625,COMPLETE
13,13,0.811470,2026-07-28 09:49:52.088851,2026-07-28 09:49:55.475832,0 days 00:00:03.386981,0.662837,0.055124,8,25,331,10,1.876296,9.985909,0.702201,4,0.038761,0.004508,0.850232,0.001685,COMPLETE
1,1,0.810783,2026-07-28 09:49:08.892292,2026-07-28 09:49:14.243652,0 days 00:00:05.351360,0.424281,0.036569,7,39,380,49,0.021917,1.824837,0.877554,5,0.082289,0.005608,0.893072,0.002576,COMPLETE


In [11]:
lgb_study.best_params

{'n_estimators': 319,
 'learning_rate': 0.05916006158291774,
 'max_depth': 6,
 'num_leaves': 10,
 'min_child_samples': 30,
 'reg_alpha': 3.6202718922173323,
 'reg_lambda': 9.996470826535594,
 'subsample': 0.7196113978680685,
 'subsample_freq': 4,
 'colsample_bytree': 0.6547144608827793}